<a href="https://colab.research.google.com/github/DaveWells-C/Soccer-Analytics/blob/main/notebooks/Day_9_OCSC_Goals_Above_xG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Orange County SC — 2026 Finishing & Expected Threat (`xG` / `xA`) Analysis
**Tools:** Python, `pandas`, `itscalledsoccer` API, Google Colab | **Source:** American Soccer Analysis (USL Championship)

## Objective
Evaluate Orange County SC players (minimum **450 minutes** played) across three lenses:
1. **Finishing Efficiency (`goals_minus_xg`):** Identifies clinical over-performers vs. under-performers relative to shot quality.
2. **Playmaking Value (`assists_minus_xa`):** Measures actual primary assists against Expected Assists (`xA`).
3. **Per-90 Expected Threat (`xGA_per_90`):** Normalizes total `xG + xA` per 90 minutes to compare high-minute starters and rotation attackers fairly.

In [1]:
!pip install -q itscalledsoccer

import pandas as pd
from itscalledsoccer.client import AmericanSoccerAnalysis

asa = AmericanSoccerAnalysis()

# Pull 2026 Orange County SC player xGoals and player names
ocsc_xg_raw = asa.get_player_xgoals(leagues="uslc", team_names="Orange County", season_name="2026")
uslc_players = asa.get_players(leagues="uslc")

if not isinstance(ocsc_xg_raw, pd.DataFrame):
    ocsc_xg_raw = ocsc_xg_raw.to_pandas()
if not isinstance(uslc_players, pd.DataFrame):
    uslc_players = uslc_players.to_pandas()

# Merge names onto stats
df = pd.merge(
    ocsc_xg_raw,
    uslc_players[["player_id", "player_name"]],
    on="player_id",
    how="left"
)

print(f"Loaded {len(df)} Orange County SC players for 2026!")

Loaded 30 Orange County SC players for 2026!


In [6]:
# Filter for players with at least 450 minutes (5 full matches)
regulars = df[df["minutes_played"] >= 450].copy()

# Calculate Goals Above Expected (goals - xgoals)
regulars["goals_minus_xg"] = regulars["goals"] - regulars["xgoals"]

# Sort from highest over-performer to lowest under-performer
finishing_table = (
    regulars[["player_name", "general_position", "minutes_played", "shots", "goals", "xgoals", "goals_minus_xg"]]
    .sort_values(by="goals_minus_xg", ascending=False)
    .round(2)
)

display(finishing_table)

,player_name,general_position,minutes_played,shots,goals,xgoals,goals_minus_xg
11,Stephen Kelly,CM,2597,26,3,1.6174,1.3826
9,Yaniv Bazini,ST,1876,46,10,8.7796,1.2204
5,Nico Benalcazar,FB,2483,15,3,2.1318,0.8682
19,Christopher Hegardt,W,2561,43,5,4.1377,0.8623
26,Tyson Espy,CB,1031,5,1,0.2903,0.7097
28,Garrison Tubbs,CB,2138,10,2,1.4753,0.5247
15,Efren Solis,DM,1041,3,1,0.7884,0.2116
21,Alex Rando,GK,2989,0,0,0,0
1,Jaylin Lindsey,FB,523,0,0,0,0
3,Ousmane Sylla,AM,2163,33,6,6.0047,-0.0047


In [7]:
def tag_finisher(diff):
    if diff >= 1.5:
        return "Clinical Over-Performer"
    elif diff <= -1.5:
        return "Unlucky / Under-Performer"
    else:
        return "Right on Expected Pace"

# Apply the function across every row of the 'goals_minus_xg' column
finishing_table["Finishing_Profile"] = finishing_table["goals_minus_xg"].apply(tag_finisher)

display(finishing_table)

,player_name,general_position,minutes_played,shots,goals,xgoals,goals_minus_xg,Finishing_Profile
11,Stephen Kelly,CM,2597,26,3,1.6174,1.3826,Right on Expected Pace
9,Yaniv Bazini,ST,1876,46,10,8.7796,1.2204,Right on Expected Pace
5,Nico Benalcazar,FB,2483,15,3,2.1318,0.8682,Right on Expected Pace
19,Christopher Hegardt,W,2561,43,5,4.1377,0.8623,Right on Expected Pace
26,Tyson Espy,CB,1031,5,1,0.2903,0.7097,Right on Expected Pace
28,Garrison Tubbs,CB,2138,10,2,1.4753,0.5247,Right on Expected Pace
15,Efren Solis,DM,1041,3,1,0.7884,0.2116,Right on Expected Pace
21,Alex Rando,GK,2989,0,0,0,0,Right on Expected Pace
1,Jaylin Lindsey,FB,523,0,0,0,0,Right on Expected Pace
3,Ousmane Sylla,AM,2163,33,6,6.0047,-0.0047,Right on Expected Pace


In [10]:
regulars["assists_minus_xa"] = regulars["primary_assists"] - regulars["xassists"]
regulars["xG_plus_xA"] = regulars["xgoals"] + regulars["xassists"]

playmaking_table = (
  regulars[["player_name", "general_position", "minutes_played", "primary_assists", "xassists", "assists_minus_xa", "xG_plus_xA"]]
  .sort_values(by="xG_plus_xA", ascending=False)
  .round(2)
)

display(playmaking_table)

,player_name,general_position,minutes_played,primary_assists,xassists,assists_minus_xa,xG_plus_xA
9,Yaniv Bazini,ST,1876,1,1.7153,-0.7153,10.4949
2,Lyam MacKinnon,W,2015,0,2.3663,-2.3663,10.1297
3,Ousmane Sylla,AM,2163,1,3.2104,-2.2104,9.2151
19,Christopher Hegardt,W,2561,5,4.9804,0.0196,9.1181
11,Stephen Kelly,CM,2597,1,4.2498,-3.2498,5.8672
22,Marcelo Palomino,CM,1144,3,2.5764,0.4236,3.3357
5,Nico Benalcazar,FB,2483,1,1.132,-0.132,3.2638
8,Ryan Doghman,FB,1893,2,2.3765,-0.3765,2.5693
10,Mouhamadou War,FB,1105,1,1.266,-0.266,1.6898
12,Brandon Cambridge,W,450,1,0.8905,0.1095,1.5724


In [14]:
regulars["xGA_per_90"] = (regulars["xG_plus_xA"] / regulars["minutes_played"]) * 90

rate_leaders = (
    regulars[["player_name", "general_position", "minutes_played", "xG_plus_xA", "xGA_per_90"]]
    .sort_values(by="xGA_per_90", ascending=False)
    .round(2)
)

rate_leaders.to_csv("ocsc_2026_xga_per90.csv", index=False)
display(rate_leaders)


,player_name,general_position,minutes_played,xG_plus_xA,xGA_per_90
9,Yaniv Bazini,ST,1876,10.4949,0.503487
2,Lyam MacKinnon,W,2015,10.1297,0.452443
3,Ousmane Sylla,AM,2163,9.2151,0.38343
19,Christopher Hegardt,W,2561,9.1181,0.320433
12,Brandon Cambridge,W,450,1.5724,0.31448
29,Jamir Johnson,W,459,1.3496,0.264627
22,Marcelo Palomino,CM,1144,3.3357,0.262424
11,Stephen Kelly,CM,2597,5.8672,0.20333
10,Mouhamadou War,FB,1105,1.6898,0.137631
8,Ryan Doghman,FB,1893,2.5693,0.122154


## Key Scouting Takeaways
* **Finishing Standout:** **[Player Name]** led the squad in `goals_minus_xg` (**+[X.XX]**), outperforming expected shot quality.
* **Chance Creation Leader:** **[Player Name]** topped total expected threat (`xG_plus_xA` = **[X.XX]**).
* **Per-90 Efficiency:** Normalizing by playing time highlighted **[Player Name]** at **[X.XX]** `xGA_per_90`